# Exploratory Data Analysis — Credit Card Fraud Detection

This notebook explores the class imbalance, transaction amount distribution, and feature correlations before any modeling happens. Run `python src/generate_synthetic_data.py` first if you don't have `data/creditcard.csv`.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

df = pd.read_csv('../data/creditcard.csv')
df.shape

In [ ]:
df.head()

## Class imbalance

This is the single most important fact about this dataset — it drives every downstream modeling decision.

In [ ]:
class_counts = df['Class'].value_counts()
fraud_rate = df['Class'].mean() * 100
print(class_counts)
print(f"Fraud rate: {fraud_rate:.4f}%")

fig, ax = plt.subplots(figsize=(5, 4))
sns.barplot(x=class_counts.index, y=class_counts.values, ax=ax)
ax.set_yscale('log')
ax.set_xticklabels(['Not Fraud', 'Fraud'])
ax.set_ylabel('Count (log scale)')
ax.set_title('Class Distribution')
plt.show()

## Transaction amount by class

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(x='Class', y='Amount', data=df, ax=axes[0])
axes[0].set_yscale('log')
axes[0].set_title('Amount by Class (log scale)')

sns.histplot(df['Amount'], bins=50, ax=axes[1])
axes[1].set_title('Overall Amount Distribution')
plt.tight_layout()
plt.show()

df.groupby('Class')['Amount'].describe()

## Transaction time patterns

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for cls, label, color in [(0, 'Not Fraud', 'steelblue'), (1, 'Fraud', 'crimson')]:
    subset = df[df['Class'] == cls]
    ax.hist(subset['Time'], bins=48, alpha=0.5, label=label, color=color, density=True)
ax.set_xlabel('Time (seconds since first transaction)')
ax.set_ylabel('Density')
ax.legend()
ax.set_title('Transaction Timing by Class')
plt.show()

## Correlation of PCA features with fraud

Since V1-V28 are PCA components, individual correlations with the target are informative even without knowing what each component originally represented.

In [ ]:
pca_cols = [c for c in df.columns if c.startswith('V')]
corrs = df[pca_cols + ['Class']].corr()['Class'].drop('Class').sort_values()

fig, ax = plt.subplots(figsize=(8, 8))
corrs.plot(kind='barh', ax=ax)
ax.set_title('Correlation of PCA Features with Fraud (Class)')
ax.set_xlabel('Pearson correlation')
plt.tight_layout()
plt.show()

print("Most positively correlated:", corrs.tail(3).index.tolist())
print("Most negatively correlated:", corrs.head(3).index.tolist())

## Takeaways

- Fraud is extremely rare — any model evaluation must use PR-AUC / recall-at-precision rather than accuracy.
- A handful of PCA features show much stronger correlation with fraud than the rest, suggesting tree-based models (which can exploit nonlinear interactions among a few informative features) may do well.
- Amount distributions differ between classes but overlap heavily — Amount alone won't separate fraud from non-fraud.

Next: `src/train.py` for model training.